In [5]:
# =========================================================
# TOURISM RATING PREDICTION MODEL
# FINAL CORRECT TRAINING PIPELINE
# =========================================================

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

# =========================================================
# LOAD DATASETS
# =========================================================

continent_df = pd.read_excel("DATA SET/Continent.xlsx")

country_df = pd.read_excel("DATA SET/Country.xlsx")

city_df = pd.read_excel("DATA SET/City.xlsx")

user_df = pd.read_excel("DATA SET/User.xlsx")

mode_df = pd.read_excel("DATA SET/Mode.xlsx")

transaction_df = pd.read_excel("DATA SET/Transaction.xlsx")

updated_item_df = pd.read_excel("DATA SET/Updated_Item.xlsx")

# =========================================================
# CLEAN COLUMN NAMES
# =========================================================

dfs = [

    continent_df,

    country_df,

    city_df,

    user_df,

    mode_df,

    transaction_df,

    updated_item_df
]

for df in dfs:

    df.columns = df.columns.str.strip()

# =========================================================
# CONVERT IDS
# =========================================================

transaction_df["UserId"] = (
    transaction_df["UserId"]
    .astype(str)
)

user_df["UserId"] = (
    user_df["UserId"]
    .astype(str)
)

transaction_df["AttractionId"] = (
    transaction_df["AttractionId"]
    .astype(str)
)

updated_item_df["AttractionId"] = (
    updated_item_df["AttractionId"]
    .astype(str)
)

transaction_df["VisitMode"] = (
    transaction_df["VisitMode"]
    .astype(str)
    .str.zfill(2)
)

mode_df["VisitModeId"] = (
    mode_df["VisitModeId"]
    .astype(str)
    .str.zfill(2)
)

# =========================================================
# MERGE COUNTRY + CONTINENT
# =========================================================

merged_country = country_df.merge(

    continent_df,

    left_on="RegionId",

    right_on="ContinentId",

    how="left"
)

# =========================================================
# MERGE CITY + COUNTRY + CONTINENT
# =========================================================

merged_city = city_df.merge(

    merged_country,

    on="CountryId",

    how="left"
)

# =========================================================
# MERGE USER DATA
# =========================================================

user_location_df = user_df.merge(

    merged_city[
        [
            "CityId",
            "CityName",
            "Country",
            "Continent"
        ]
    ],

    on="CityId",

    how="left"
)

# =========================================================
# MERGE VISIT MODE
# =========================================================

transaction_df = transaction_df.merge(

    mode_df,

    left_on="VisitMode",

    right_on="VisitModeId",

    how="left"
)

# =========================================================
# MERGE TRANSACTION + USER LOCATION
# =========================================================

final_df = transaction_df.merge(

    user_location_df[
        [
            "UserId",
            "Continent",
            "Country",
            "CityName"
        ]
    ],

    on="UserId",

    how="left"
)

# =========================================================
# MERGE ATTRACTION DETAILS
# =========================================================

final_df = final_df.merge(

    updated_item_df[
        [
            "AttractionId",
            "AttractionType"
        ]
    ],

    on="AttractionId",

    how="left"
)

# =========================================================
# REQUIRED COLUMNS
# =========================================================

required_columns = [

    "Continent",

    "Country",

    "CityName",

    "VisitYear",

    "VisitMonth",

    "VisitMode_y",

    "AttractionType",

    "Rating"
]

final_df = final_df[required_columns]

# =========================================================
# REMOVE NULLS
# =========================================================

final_df = final_df.dropna()

# =========================================================
# RENAME COLUMN
# =========================================================

final_df.rename(

    columns={
        "VisitMode_y": "VisitMode"
    },

    inplace=True
)

# =========================================================
# FEATURES
# =========================================================

X = final_df[
    [
        "Continent",
        "Country",
        "CityName",
        "VisitYear",
        "VisitMonth",
        "VisitMode",
        "AttractionType"
    ]
]

# =========================================================
# TARGET
# =========================================================

y = final_df["Rating"]

# =========================================================
# CATEGORICAL FEATURES
# =========================================================

categorical_features = [

    "Continent",

    "Country",

    "CityName",

    "VisitMode",

    "AttractionType"
]

# =========================================================
# PREPROCESSOR
# =========================================================

preprocessor = ColumnTransformer(

    transformers=[

        (

            "cat",

            OneHotEncoder(
                handle_unknown="ignore"
            ),

            categorical_features
        )
    ],

    remainder="passthrough"
)

# =========================================================
# MODEL PIPELINE
# =========================================================

model = Pipeline([

    (

        "preprocessor",

        preprocessor
    ),

    (

        "regressor",

        RandomForestRegressor(

            n_estimators=300,

            random_state=42
        )
    )
])

# =========================================================
# TRAIN TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.2,

    random_state=42
)

# =========================================================
# TRAIN MODEL
# =========================================================

model.fit(

    X_train,

    y_train
)

# =========================================================
# PREDICT
# =========================================================

y_pred = model.predict(X_test)

# =========================================================
# EVALUATION
# =========================================================

mae = mean_absolute_error(

    y_test,

    y_pred
)

r2 = r2_score(

    y_test,

    y_pred
)

print("\n======================")
print("MODEL PERFORMANCE")
print("======================")

print("MAE :", round(mae, 2))

print("R2 Score :", round(r2, 2))

# =========================================================
# SAVE MODEL
# =========================================================

joblib.dump(

    model,

    "rating_prediction_model.pkl"
)

print("\nModel Saved Successfully")


MODEL PERFORMANCE
MAE : 0.86
R2 Score : -0.07

Model Saved Successfully


In [17]:
# =========================================================
# FINAL HIGH ACCURACY CLASSIFICATION TRAINING
# TOURISM EXPERIENCE ANALYTICS
# TARGET = VISIT MODE
# =========================================================

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import VotingClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

from sklearn.utils import resample

# =========================================================
# LOAD DATASETS
# =========================================================

continent_df = pd.read_excel(
    "DATA SET/Continent.xlsx"
)

region_df = pd.read_excel(
    "DATA SET/Region.xlsx"
)

country_df = pd.read_excel(
    "DATA SET/Country.xlsx"
)

city_df = pd.read_excel(
    "DATA SET/City.xlsx"
)

user_df = pd.read_excel(
    "DATA SET/User.xlsx"
)

mode_df = pd.read_excel(
    "DATA SET/Mode.xlsx"
)

transaction_df = pd.read_excel(
    "DATA SET/Transaction.xlsx"
)

updated_item_df = pd.read_excel(
    "DATA SET/Updated_Item.xlsx"
)

# =========================================================
# CLEAN COLUMN NAMES
# =========================================================

dfs = [

    continent_df,
    region_df,
    country_df,
    city_df,
    user_df,
    mode_df,
    transaction_df,
    updated_item_df
]

for df in dfs:

    df.columns = df.columns.str.strip()

# =========================================================
# CONVERT IDS
# =========================================================

transaction_df["UserId"] = (
    transaction_df["UserId"]
    .astype(str)
)

user_df["UserId"] = (
    user_df["UserId"]
    .astype(str)
)

transaction_df["AttractionId"] = (
    transaction_df["AttractionId"]
    .astype(str)
)

updated_item_df["AttractionId"] = (
    updated_item_df["AttractionId"]
    .astype(str)
)

transaction_df["VisitMode"] = (
    transaction_df["VisitMode"]
    .astype(str)
    .str.zfill(2)
)

mode_df["VisitModeId"] = (
    mode_df["VisitModeId"]
    .astype(str)
    .str.zfill(2)
)

# =========================================================
# MERGE COUNTRY + REGION
# =========================================================

merged_country = country_df.merge(

    region_df,

    on="RegionId",

    how="left"
)

# =========================================================
# MERGE CONTINENT
# =========================================================

merged_country = merged_country.merge(

    continent_df,

    on="ContinentId",

    how="left"
)

# =========================================================
# MERGE CITY
# =========================================================

merged_city = city_df.merge(

    merged_country,

    on="CountryId",

    how="left"
)

# =========================================================
# MERGE USER LOCATION
# =========================================================

user_location_df = user_df.merge(

    merged_city[
        [
            "CityId",
            "CityName",
            "Country",
            "Region",
            "Continent"
        ]
    ],

    on="CityId",

    how="left"
)

# =========================================================
# MERGE VISIT MODE
# =========================================================

transaction_df = transaction_df.merge(

    mode_df,

    left_on="VisitMode",

    right_on="VisitModeId",

    how="left"
)

# =========================================================
# MERGE TRANSACTION + USER LOCATION
# =========================================================

final_df = transaction_df.merge(

    user_location_df[
        [
            "UserId",
            "Continent",
            "Region",
            "Country",
            "CityName"
        ]
    ],

    on="UserId",

    how="left"
)

# =========================================================
# MERGE ATTRACTION DETAILS
# =========================================================

final_df = final_df.merge(

    updated_item_df[
        [
            "AttractionId",
            "AttractionType"
        ]
    ],

    on="AttractionId",

    how="left"
)

# =========================================================
# REQUIRED COLUMNS
# =========================================================

required_columns = [

    "Continent",

    "Region",

    "Country",

    "CityName",

    "VisitYear",

    "VisitMonth",

    "VisitMode_y",

    "AttractionType"
]

final_df = final_df[required_columns]

# =========================================================
# REMOVE NULL VALUES
# =========================================================

final_df = final_df.dropna()

# =========================================================
# REMOVE INVALID VALUES
# =========================================================

final_df = final_df[
    (final_df["Continent"] != "-")
    &
    (final_df["Region"] != "-")
    &
    (final_df["Country"] != "-")
    &
    (final_df["CityName"] != "-")
]

# =========================================================
# RENAME TARGET
# =========================================================

final_df.rename(

    columns={
        "VisitMode_y": "VisitMode"
    },

    inplace=True
)

# =========================================================
# DISPLAY CLASS DISTRIBUTION
# =========================================================

print("\n========================")
print("ORIGINAL CLASS DISTRIBUTION")
print("========================")

print(
    final_df["VisitMode"]
    .value_counts()
)

# =========================================================
# BALANCE DATASET
# =========================================================

max_count = final_df[
    "VisitMode"
].value_counts().max()

balanced_data = []

for label in final_df["VisitMode"].unique():

    subset = final_df[
        final_df["VisitMode"] == label
    ]

    upsampled = resample(

        subset,

        replace=True,

        n_samples=max_count,

        random_state=42
    )

    balanced_data.append(
        upsampled
    )

final_df = pd.concat(
    balanced_data
)

# =========================================================
# SHUFFLE DATA
# =========================================================

final_df = final_df.sample(

    frac=1,

    random_state=42
)

# =========================================================
# FEATURES
# =========================================================

X = final_df[
    [
        "Continent",
        "Region",
        "Country",
        "CityName",
        "VisitYear",
        "VisitMonth",
        "AttractionType"
    ]
]

# =========================================================
# TARGET
# =========================================================

y = final_df["VisitMode"]

# =========================================================
# LABEL ENCODING
# =========================================================

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

# =========================================================
# LABEL MAPPING
# =========================================================

print("\n========================")
print("LABEL MAPPING")
print("========================")

for idx, label in enumerate(
    label_encoder.classes_
):

    print(idx, "→", label)

# =========================================================
# CATEGORICAL FEATURES
# =========================================================

categorical_features = [

    "Continent",

    "Region",

    "Country",

    "CityName",

    "AttractionType"
]

# =========================================================
# PREPROCESSOR
# =========================================================

preprocessor = ColumnTransformer(

    transformers=[

        (

            "cat",

            OneHotEncoder(
                handle_unknown="ignore"
            ),

            categorical_features
        )
    ],

    remainder="passthrough"
)

# =========================================================
# ENSEMBLE MODELS
# =========================================================

rf_model = RandomForestClassifier(

    n_estimators=120,

    max_depth=15,

    min_samples_split=3,

    class_weight="balanced",

    random_state=42
)

et_model = ExtraTreesClassifier(

    n_estimators=120,

    max_depth=15,

    class_weight="balanced",

    random_state=42
)

ensemble_model = VotingClassifier(

    estimators=[

        ("rf", rf_model),

        ("et", et_model)
    ],

    voting="soft"
)

# =========================================================
# FINAL PIPELINE
# =========================================================

model = Pipeline([

    (

        "preprocessor",

        preprocessor
    ),

    (

        "classifier",

        ensemble_model
    )
])

# =========================================================
# TRAIN TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y_encoded,

    test_size=0.2,

    random_state=42,

    stratify=y_encoded
)

# =========================================================
# TRAIN MODEL
# =========================================================

model.fit(

    X_train,

    y_train
)

# =========================================================
# PREDICTIONS
# =========================================================

y_pred = model.predict(
    X_test
)

# =========================================================
# ACCURACY
# =========================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

print("\n========================")
print("MODEL ACCURACY")
print("========================")

print(
    round(accuracy * 100, 2),
    "%"
)

print("\n========================")
print("CLASSIFICATION REPORT")
print("========================")

print(
    classification_report(
        y_test,
        y_pred
    )
)

# =========================================================
# SAVE MODEL
# =========================================================

joblib.dump(

    model,

    "visitmode_prediction_model.pkl",

    compress=3
)

# =========================================================
# SAVE LABEL ENCODER
# =========================================================

joblib.dump(

    label_encoder,

    "label_encoder.pkl"
)

# =========================================================
# SAVE FEATURE COLUMNS
# =========================================================

feature_columns = list(X.columns)

joblib.dump(

    feature_columns,

    "feature_columns.pkl"
)

# =========================================================
# SUCCESS
# =========================================================

print("\n========================")
print("PKL FILES SAVED")
print("========================")

print("visitmode_prediction_model.pkl")

print("label_encoder.pkl")

print("feature_columns.pkl")


ORIGINAL CLASS DISTRIBUTION
VisitMode
Couples     21603
Family      15208
Friends     10939
Solo         4517
Business      623
Name: count, dtype: int64

LABEL MAPPING
0 → Business
1 → Couples
2 → Family
3 → Friends
4 → Solo

MODEL ACCURACY
44.26 %

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.50      0.80      0.61      4321
           1       0.38      0.46      0.42      4320
           2       0.38      0.41      0.40      4321
           3       0.71      0.13      0.22      4321
           4       0.44      0.41      0.43      4320

    accuracy                           0.44     21603
   macro avg       0.48      0.44      0.41     21603
weighted avg       0.48      0.44      0.41     21603


PKL FILES SAVED
visitmode_prediction_model.pkl
label_encoder.pkl
feature_columns.pkl
